# FlyLM — 뉴런 16개부터 검증

**목적:** attention 없이, 고정된 유향 연결 위의 스칼라 recurrent state로 next-token prediction을 구현한다.
기본값은 **16 neurons**, 뉴런별 상태 1개, 학습되는 edge weight, 입력/출력 adapter이다.
실제 MaleCNS 연결 및 한국어 JSON 학습 데이터는 이번 단계에 넣지 않는다.
기본 그래프는 합성이며 실제 초파리 커넥톰 또는 생물학적 우수성의 검증으로 해석하면 안 된다.

**실행:** Colab 등에서 T4 GPU 런타임 선택 → 위에서부터 실행. PyTorch와 NumPy가 있는 환경을 사용한다.
추가 설치·다운로드는 하지 않는다. 이 파일은 로컬에서 실행하지 않고 구조·문법만 cold test했다.
아래 검증/학습 셀은 사용자가 원격에서 실행할 코드이며, 저장된 실행 결과는 없다.

검증 순서: sparse 연산과 dense 기준 비교 → gradient/인과성/streaming 검사 → 작은 고정 패턴 과적합 → 메모리·처리량 확인.
16개 모델은 GPU 활용률이 낮아 T4의 최대 성능이나 대규모 모델 성능을 예측하는 벤치마크가 아니다.


In [ ]:
import copy
import math
import time
import random
import platform
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

CFG = dict(seed=42, n_neurons=16, embedding_dim=8, n_inputs=4,
           n_outputs=4, brain_steps=2, batch_size=4, seq_len=16,
           train_steps=300, lr=0.01, log_every=50)
random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# FP32: 이 크기에서는 AMP가 불필요하며 T4용 BF16을 전제하지 않는다.
# CUDA index_add는 비결정적일 수 있으므로 동일 seed가 bitwise 재현을 보장하지 않는다.
print(dict(python=platform.python_version(), torch=torch.__version__,
           device=str(device), gpu=torch.cuda.get_device_name(0) if device.type == 'cuda' else None))
if device.type != 'cuda':
    print('CPU fallback입니다. T4 메모리/속도 결과는 얻을 수 없습니다.')
assert CFG['n_neurons'] == 16, '첫 실험은 16개 기준입니다. 확대 전 구조와 메모리 계획을 재검토하세요.'


## 고정 그래프와 상태 갱신

합성 그래프는 각 노드에서 다음 노드와 4칸 뒤 노드로 향하는 32개 edge를 갖는다.
`src → dst` 방향으로만 메시지를 더한다. 모든 노드가 ring으로 연결되며 self-edge는 없다.
입력 포트 0–3과 출력 포트 4–7은 실험용 지정으로, sensory/motor annotation이 아니다.

각 brain step은 `h_new = 0.5*h + 0.5*tanh(sum(w*h_src) + input + bias)`이다.
0.5의 누설 상태 유지는 뉴런 자신의 시간 기억이며 추가적인 뉴런 간 edge는 아니다.
입력은 토큰당 두 번 주입하고, 마지막 상태에서 다음 토큰 logits를 읽는다.
학습되는 연결은 edge 목록에 존재하는 32개뿐이다. 초기 가중치는 incoming degree로 정규화한다.


In [ ]:
def make_graph(n):
    pairs = sorted({(i, (i + offset) % n) for i in range(n) for offset in (1, 4)})
    src, dst = torch.tensor(pairs, dtype=torch.long).T
    return src, dst

src, dst = make_graph(CFG['n_neurons'])
input_nodes = torch.arange(CFG['n_inputs'])
output_nodes = torch.arange(4, 4 + CFG['n_outputs'])
print(dict(neurons=CFG['n_neurons'], edges=src.numel(),
           input_nodes=input_nodes.tolist(), output_nodes=output_nodes.tolist(),
           topology='synthetic ring + offset-4'))


## 모델

UTF-8 byte vocabulary 256개에 BOS/EOS를 더해 258개를 사용한다.
한국어도 손실 없이 인코딩하지만 한글 한 글자가 여러 byte가 되므로 추후 BPE 대비 문맥 효율은 낮다.
decoder는 출력 뉴런 → 8차원 bottleneck → tied embedding이다. 입력에서 decoder로 직접 우회하는 경로는 없다.
희소 계산은 gather + index_add로 구현한다. 모델은 N×N dense adjacency를 만들지 않는다.


In [ ]:
BOS, EOS, VOCAB = 256, 257, 258

def encode(text):
    return [BOS] + list(text.encode('utf-8')) + [EOS]

def decode(ids):
    # 생성 중 불완전한 UTF-8은 replacement character로 표시한다.
    return bytes(i for i in ids if 0 <= i < 256).decode('utf-8', errors='replace')

class FlyLM(nn.Module):
    def __init__(self, cfg, src, dst, input_nodes, output_nodes):
        super().__init__()
        self.n = cfg['n_neurons']
        self.k = cfg['brain_steps']
        for name, value in [('src', src), ('dst', dst),
                            ('input_nodes', input_nodes), ('output_nodes', output_nodes)]:
            self.register_buffer(name, value.clone())
        degree = torch.bincount(dst, minlength=self.n).float().clamp_min(1)
        self.edge_weight = nn.Parameter(torch.randn(src.numel()) * 0.5 / degree[dst].sqrt())
        self.bias = nn.Parameter(torch.zeros(self.n))
        self.embedding = nn.Embedding(VOCAB, cfg['embedding_dim'])
        self.input_adapter = nn.Linear(cfg['embedding_dim'], input_nodes.numel())
        self.output_adapter = nn.Linear(output_nodes.numel(), cfg['embedding_dim'])
        self.output_bias = nn.Parameter(torch.zeros(VOCAB))

    def aggregate(self, h):
        messages = h.index_select(1, self.src) * self.edge_weight
        return torch.zeros_like(h).index_add(1, self.dst, messages)

    def forward(self, tokens, state=None):
        if tokens.ndim != 2 or tokens.shape[1] == 0:
            raise ValueError('tokens must be [batch, nonempty time]')
        batch, length = tokens.shape
        h = self.embedding.weight.new_zeros(batch, self.n) if state is None else state
        if h.shape != (batch, self.n):
            raise ValueError('state shape mismatch')
        logits = []
        for t in range(length):
            injected = self.input_adapter(self.embedding(tokens[:, t]))
            drive = torch.zeros_like(h).index_add(1, self.input_nodes, injected)
            for _ in range(self.k):
                h = 0.5 * h + 0.5 * torch.tanh(self.aggregate(h) + drive + self.bias)
            z = self.output_adapter(h.index_select(1, self.output_nodes))
            logits.append(F.linear(z, self.embedding.weight, self.output_bias))
        return torch.stack(logits, dim=1), h

model = FlyLM(CFG, src, dst, input_nodes, output_nodes).to(device)
print('parameters:', sum(p.numel() for p in model.parameters()))
print('trainable edge weights:', model.edge_weight.numel())
assert decode(encode('안녕하세요')) == '안녕하세요'


## 원격 실행용 구조 검증

16×16 dense 행렬은 아래 정답 비교에서만 만든다. 순전파뿐 아니라 edge/state gradient도 비교한다.
미래 토큰 변경이 과거 logits에 영향을 주지 않는지, chunk 분할과 단일 실행이 같은지,
실제로 과거 상태를 사용하는지, 한 번의 optimizer step이 edge를 갱신하는지 확인한다.
이 검사는 학습 전 모델의 복사본을 사용한다.


In [ ]:
def check_model(original):
    m = copy.deepcopy(original)
    h = torch.randn(2, m.n, device=device, requires_grad=True)
    sparse = m.aggregate(h)
    dense_w = torch.zeros(m.n, m.n, device=device).index_put((m.dst, m.src), m.edge_weight)
    dense = h @ dense_w.T
    torch.testing.assert_close(sparse, dense, rtol=1e-5, atol=1e-6)
    gs = torch.autograd.grad(sparse.square().sum(), (h, m.edge_weight), retain_graph=True)
    gd = torch.autograd.grad(dense.square().sum(), (h, m.edge_weight))
    for a, b in zip(gs, gd):
        torch.testing.assert_close(a, b, rtol=1e-5, atol=1e-6)

    x = torch.randint(0, 256, (2, 8), device=device)
    with torch.no_grad():
        full, final = m(x)
        first, state = m(x[:, :4])
        second, end = m(x[:, 4:], state)
        torch.testing.assert_close(full, torch.cat([first, second], dim=1))
        torch.testing.assert_close(final, end)
        changed = x.clone()
        changed[:, 4:] = (changed[:, 4:] + 1) % 256
        altered, _ = m(changed)
        torch.testing.assert_close(full[:, :4], altered[:, :4])
        reset, _ = m(x[:, 4:])
        assert (second - reset).abs().max().item() > 1e-7, 'state has no measurable effect'

    before = m.edge_weight.detach().clone()
    edge_src, edge_dst = m.src.clone(), m.dst.clone()
    optimizer = torch.optim.Adam(m.parameters(), lr=CFG['lr'])
    logits, _ = m(x)
    loss = F.cross_entropy(logits[:, :-1].reshape(-1, VOCAB), x[:, 1:].reshape(-1))
    loss.backward()
    for name, p in m.named_parameters():
        assert p.grad is not None and torch.isfinite(p.grad).all(), name
    assert m.edge_weight.grad.abs().sum().item() > 0
    optimizer.step()
    assert not torch.equal(before, m.edge_weight)
    assert torch.equal(edge_src, m.src) and torch.equal(edge_dst, m.dst)
    assert not m.src.requires_grad and not m.dst.requires_grad
    print('PASS: sparse/dense values + gradients, causality, streaming, state, edge update, fixed topology')

check_model(model)


## 작은 패턴 과적합 — 실제 코퍼스 학습 아님

내장 문자열 `가나다.`를 반복한 고정 batch에 next-byte prediction을 학습한다.
같은 batch의 loss 감소/정확도는 optimizer 작동 확인일 뿐 일반화나 한국어 이해를 의미하지 않는다.
각 샘플은 독립적으로 zero state에서 시작하며, 길이 16의 범위 안에서 BPTT를 수행한다.
별도의 TBPTT 구현은 아직 필요하지 않다. 추후 연속 문서에서는 반환 state를 유지하되 chunk 사이에서
`state = state.detach()`하고, 문서/배치 경계에서는 reset해야 한다.


In [ ]:
pattern = list('가나다.'.encode('utf-8'))
stream = torch.tensor(pattern * 20, dtype=torch.long, device=device)
windows = torch.stack([stream[i:i + CFG['seq_len'] + 1] for i in range(CFG['batch_size'])])
x_train, y_train = windows[:, :-1], windows[:, 1:]

@torch.no_grad()
def evaluate(m):
    m.eval()
    logits, _ = m(x_train)
    loss = F.cross_entropy(logits.reshape(-1, VOCAB), y_train.reshape(-1)).item()
    accuracy = (logits.argmax(-1) == y_train).float().mean().item()
    return dict(loss=loss, accuracy=accuracy)

# 이 셀을 재실행해도 동일한 초기화에서 시작한다.
torch.manual_seed(CFG['seed'])
model = FlyLM(CFG, src, dst, input_nodes, output_nodes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=CFG['lr'])
initial = evaluate(model)
if device.type == 'cuda':
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
history = []
for step in range(1, CFG['train_steps'] + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    logits, _ = model(x_train)
    loss = F.cross_entropy(logits.reshape(-1, VOCAB), y_train.reshape(-1))
    if not torch.isfinite(loss):
        raise RuntimeError(f'Non-finite loss at step {step}')
    loss.backward()
    grad_norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
    optimizer.step()
    if step == 1 or step % CFG['log_every'] == 0:
        row = dict(step=step, **evaluate(model), grad_norm=float(grad_norm))
        history.append(row)
        print(row)
if device.type == 'cuda':
    torch.cuda.synchronize()
elapsed = time.perf_counter() - started
final = evaluate(model)
report = dict(initial=initial, final=final,
              loss_reduction_fraction=1 - final['loss'] / initial['loss'],
              elapsed_seconds=elapsed,
              training_tokens_per_second=CFG['train_steps'] * x_train.numel() / elapsed,
              peak_allocated_MiB=torch.cuda.max_memory_allocated() / 2**20 if device.type == 'cuda' else None,
              peak_reserved_MiB=torch.cuda.max_memory_reserved() / 2**20 if device.type == 'cuda' else None)
print(report)
print('과적합 기준:', 'PASS' if final['loss'] < initial['loss'] * 0.5 and final['accuracy'] >= 0.9 else 'NOT MET')


## 추론 및 결과 해석

패턴 내부에서 학습했으므로 동일한 패턴의 prefix로 이어 쓴다. BOS/EOS 예측은 이번 과적합에서 학습하지 않았다.
구조 검사 통과 + 유한한 gradient + loss 50% 이상 감소 + 고정 batch 정확도 90% 이상을 첫 관문으로 삼는다.
기준 미달이면 뉴런을 늘리기 전에 출력 포트/학습률/갱신 횟수와 loss 추이를 확인한다.
처리량에는 평가와 Python loop 비용이 포함된다. GPU 메모리는 PyTorch allocator 기준이며 전체 GPU 사용량은 아니다.


In [ ]:
@torch.no_grad()
def generate(m, prefix='가', max_new_bytes=30):
    m.eval()
    ids = list(prefix.encode('utf-8'))
    if not ids:
        raise ValueError('A nonempty prefix is required for this pattern experiment')
    logits, state = m(torch.tensor([ids], device=device))
    for _ in range(max_new_bytes):
        next_id = int(logits[0, -1].argmax())
        if next_id == EOS:
            break
        ids.append(next_id)
        logits, state = m(torch.tensor([[next_id]], device=device), state)
    return decode(ids)

print(generate(model))


## 다음 단계의 경계

- 이번 파일은 16개 모델의 계산/학습 검증까지다. JSON 로더, tokenizer 학습, 실제 커넥톰 다운로드는 포함하지 않는다.
- 한국어 JSON 연결 시 실제 텍스트 필드를 명시하고 문서 단위 train/validation 분리 후 tokenizer를 선택한다.
- MaleCNS 연결을 넣을 때 source/destination ID를 재매핑하고 중복 pair의 synapse count를 합산한다.
  작은 유도 부분 그래프는 원본 전체 topology와 다르며 포트의 생물학적 annotation도 별도로 확인해야 한다.
- 실제 topology의 효과는 degree 보존 random graph 및 일반 RNN과 같은 데이터/예산으로 비교해야 한다.
- 166k 확장은 이 설정의 숫자만 바꾸는 것으로 검증되지 않는다. edge 수, B×E message 메모리,
  sequence length×brain steps의 autograd 비용, TBPTT를 먼저 측정한다.

참고: [MaleCNS 공식 다운로드](https://male-cns.janelia.org/download/)는 segment 간 전체 연결 그래프를 제공한다.
연산 API: [PyTorch index_add](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.index_add.html).
